In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pprint

# Load checkpoint
checkpoint_path = Path('rc_cnn_hierarchical_v3.pt')
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

print("Checkpoint keys:")
print(list(checkpoint.keys()))

Checkpoint keys:
['model_state_dict', 'superfamily_names', 'superfamily_to_id', 'arch', 'best_epoch']


In [2]:
# Model architecture info
print("=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
arch = checkpoint.get('arch', {})
for key, value in arch.items():
    print(f"  {key}: {value}")

print(f"\nBest epoch: {checkpoint.get('best_epoch', 'N/A')}")

MODEL ARCHITECTURE
  width: 128
  motif_kernels: (7, 15, 21)
  context_kernel: 9
  context_dilations: (1, 2, 4, 8)
  rc_mode: early
  num_superfamilies: 7

Best epoch: 19


In [3]:
# Superfamily classes
print("=" * 60)
print("SUPERFAMILY CLASSES")
print("=" * 60)
superfamily_names = checkpoint.get('superfamily_names', [])
superfamily_to_id = checkpoint.get('superfamily_to_id', {})

print(f"Number of superfamilies: {len(superfamily_names)}")
print(f"\nSuperfamily names:")
for i, name in enumerate(superfamily_names):
    print(f"  {i}: {name}")

SUPERFAMILY CLASSES
Number of superfamilies: 7

Superfamily names:
  0: DNA
  1: DNA/Academ-1
  2: DNA/CMC
  3: DNA/PIF-Harbinger
  4: DNA/PiggyBac
  5: DNA/TcMar-Tc1
  6: DNA/hAT


In [4]:
# Training history
history = checkpoint.get('history', {})
print("=" * 60)
print("TRAINING HISTORY")
print("=" * 60)
print(f"History keys: {list(history.keys())}")
print(f"Training epochs: {len(history.get('train_loss', []))}")

TRAINING HISTORY
History keys: []
Training epochs: 0


In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

epochs = range(1, len(history.get('train_loss', [])) + 1)

# Training losses
ax1 = axes[0, 0]
if 'train_loss' in history:
    ax1.plot(epochs, history['train_loss'], 'b-', label='Total Loss', linewidth=2)
if 'train_binary_loss' in history:
    ax1.plot(epochs, history['train_binary_loss'], 'g--', label='Binary Loss', linewidth=1.5)
if 'train_sf_loss' in history:
    ax1.plot(epochs, history['train_sf_loss'], 'r--', label='Superfamily Loss', linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Losses')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Binary classification metrics
ax2 = axes[0, 1]
if 'val_binary_acc' in history:
    ax2.plot(epochs, history['val_binary_acc'], 'b-', label='Accuracy', linewidth=2)
if 'val_binary_f1' in history:
    ax2.plot(epochs, history['val_binary_f1'], 'g-', label='F1 Score', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('Binary Classification (Transposase+ vs None)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1])

# Superfamily classification metrics
ax3 = axes[1, 0]
if 'val_sf_acc' in history:
    ax3.plot(epochs, history['val_sf_acc'], 'b-', label='Accuracy', linewidth=2)
if 'val_sf_f1' in history:
    ax3.plot(epochs, history['val_sf_f1'], 'g-', label='F1 Score (Macro)', linewidth=2)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Score')
ax3.set_title('Superfamily Classification (Multi-class)')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim([0, 1])

# Combined score
ax4 = axes[1, 1]
if 'val_binary_f1' in history and 'val_sf_f1' in history:
    combined = [0.5 * b + 0.5 * s for b, s in zip(history['val_binary_f1'], history['val_sf_f1'])]
    ax4.plot(epochs, combined, 'purple', label='Combined Score', linewidth=2)
    best_epoch = checkpoint.get('best_epoch', np.argmax(combined) + 1)
    ax4.axvline(x=best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Score')
ax4.set_title('Combined Score (0.5 × Binary F1 + 0.5 × SF F1)')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final performance summary
print("=" * 60)
print("FINAL PERFORMANCE (Best Epoch)")
print("=" * 60)

best_epoch = checkpoint.get('best_epoch', len(history.get('train_loss', [])))
best_idx = best_epoch - 1 if best_epoch else -1

print(f"Best epoch: {best_epoch}")
print()

if history:
    print("Binary Classification (Transposase+ vs None):")
    if 'val_binary_acc' in history:
        print(f"  Accuracy: {history['val_binary_acc'][best_idx]:.4f}")
    if 'val_binary_f1' in history:
        print(f"  F1 Score: {history['val_binary_f1'][best_idx]:.4f}")
    
    print()
    print("Superfamily Classification (Multi-class):")
    if 'val_sf_acc' in history:
        print(f"  Accuracy: {history['val_sf_acc'][best_idx]:.4f}")
    if 'val_sf_f1' in history:
        print(f"  F1 Score (Macro): {history['val_sf_f1'][best_idx]:.4f}")
    
    print()
    if 'val_binary_f1' in history and 'val_sf_f1' in history:
        combined = 0.5 * history['val_binary_f1'][best_idx] + 0.5 * history['val_sf_f1'][best_idx]
        print(f"Combined Score: {combined:.4f}")

In [5]:
# Model state dict analysis
print("=" * 60)
print("MODEL STATE DICT ANALYSIS")
print("=" * 60)

state_dict = checkpoint.get('model_state_dict', {})

# Count parameters by layer type
layer_params = {}
total_params = 0

for name, tensor in state_dict.items():
    params = tensor.numel()
    total_params += params
    
    # Get layer group
    layer_group = name.split('.')[0]
    if layer_group not in layer_params:
        layer_params[layer_group] = 0
    layer_params[layer_group] += params

print(f"Total parameters: {total_params:,}")
print()
print("Parameters by layer group:")
for group, params in sorted(layer_params.items(), key=lambda x: -x[1]):
    pct = 100 * params / total_params
    print(f"  {group}: {params:,} ({pct:.1f}%)")

MODEL STATE DICT ANALYSIS
Total parameters: 731,668

Parameters by layer group:
  context_blocks: 592,388 (81.0%)
  mix: 49,793 (6.8%)
  superfamily_head: 34,823 (4.8%)
  motif_convs: 29,443 (4.0%)
  binary_head: 16,770 (2.3%)
  boundary_head: 8,451 (1.2%)


In [ ]:
# Learning rate schedule visualization
if 'train_loss' in history:
    epochs_total = len(history['train_loss'])
    lr_base = 0.001  # from config
    lr_min = lr_base * 0.01
    
    # Cosine annealing schedule
    lr_schedule = []
    for ep in range(epochs_total):
        lr = lr_min + 0.5 * (lr_base - lr_min) * (1 + np.cos(np.pi * ep / 50))
        lr_schedule.append(lr)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, epochs_total + 1), lr_schedule, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Cosine Annealing Learning Rate Schedule')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()

In [6]:
# Summary table
print("=" * 60)
print("CHECKPOINT SUMMARY")
print("=" * 60)
print(f"Checkpoint file: {checkpoint_path}")
print(f"File size: {checkpoint_path.stat().st_size / 1024 / 1024:.2f} MB")
print()
print("Architecture:")
arch = checkpoint.get('arch', {})
print(f"  - Canvas length: {arch.get('fixed_length', 'N/A')} bp")
print(f"  - Width: {arch.get('width', 'N/A')}")
print(f"  - Motif kernels: {arch.get('motif_kernels', 'N/A')}")
print(f"  - Context dilations: {arch.get('context_dilations', 'N/A')}")
print(f"  - RC mode: {arch.get('rc_mode', 'N/A')}")
print(f"  - Num superfamilies: {arch.get('num_superfamilies', 'N/A')}")
print()
print("Training:")
print(f"  - Total epochs run: {len(history.get('train_loss', []))}")
print(f"  - Best epoch: {checkpoint.get('best_epoch', 'N/A')}")
print(f"  - Total parameters: {total_params:,}")

CHECKPOINT SUMMARY
Checkpoint file: rc_cnn_hierarchical_v3.pt
File size: 2.81 MB

Architecture:
  - Canvas length: N/A bp
  - Width: 128
  - Motif kernels: (7, 15, 21)
  - Context dilations: (1, 2, 4, 8)
  - RC mode: early
  - Num superfamilies: 7

Training:
  - Total epochs run: 0
  - Best epoch: 19
  - Total parameters: 731,668


## Model Evaluation on Test Data

Since the checkpoint doesn't include training history, let's load the model and evaluate it on the data.